# Synthetic Option Trade Dataset

**Project constraint:** This notebook uses only accessible/public information and synthetic modeling. It does not use proprietary internal trading data. Simulated trade dates run from **January 1, 2026 through May 29, 2026**.

The generator provides broad coverage across underlyings, strikes, standard monthly expiries (third Friday), maturities, trade sizes, sides, and times of day. Trade-size probabilities are a transparent public-data proxy informed by the SEC's observation that one-contract trades are material in actively traded options and by Cboe's published size categories (`<100`, `100-199`, and `>199` contracts). Exact trade-level historical size data is generally proprietary, so the assumptions below are deliberately configurable rather than presented as a fitted proprietary distribution.

Public references:
- SEC, *Staff Report on Equity and Options Market Structure Conditions in Early 2021*: https://www.sec.gov/files/staff-report-equity-options-market-struction-conditions-early-2021.pdf
- Cboe historical options volume download: https://www.cboe.com/us/options/market_statistics/historical_data/
- Cboe Open-Close Volume Summary product description and size categories: https://datashop.cboe.com/cboe-options-open-close-volume-summary

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

SEED = 20260612
N_TRADES = 50_000
START_DATE = pd.Timestamp('2026-01-01')
END_DATE = pd.Timestamp('2026-05-29')
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Run this notebook from the project root or notebooks directory.')
OUTPUT_CSV = PROJECT_ROOT / 'data' / 'simulated' / 'synthetic_option_trades_2026.csv'
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

# Liquid, student-accessible reference universe. Weights are stylized volume shares.
UNDERLYINGS = pd.DataFrame({
    'symbol': ['SPY', 'QQQ', 'NVDA', 'AAPL', 'MSFT', 'JPM'],
    'start_spot': [590.0, 515.0, 145.0, 245.0, 455.0, 250.0],
    'annual_vol': [0.18, 0.22, 0.48, 0.28, 0.25, 0.24],
    'volume_weight': [0.30, 0.20, 0.17, 0.13, 0.10, 0.10],
}).set_index('symbol')

# Explicit 2026 U.S. equity-market closures inside the study window.
MARKET_HOLIDAYS = pd.to_datetime([
    '2026-01-01', '2026-01-19', '2026-02-16', '2026-04-03', '2026-05-25'
])
business_days = pd.bdate_range(START_DATE, END_DATE)
trading_days = business_days[~business_days.isin(MARKET_HOLIDAYS)]
len(trading_days), trading_days.min(), trading_days.max()

(102, Timestamp('2026-01-02 00:00:00'), Timestamp('2026-05-29 00:00:00'))

In [2]:
def third_friday(year, month):
    first = pd.Timestamp(year=year, month=month, day=1)
    first_friday = first + pd.Timedelta(days=(4 - first.weekday()) % 7)
    return first_friday + pd.Timedelta(days=14)


def monthly_expiries(start='2026-01-01', end='2027-06-30'):
    months = pd.period_range(start, end, freq='M')
    return pd.DatetimeIndex([third_friday(p.year, p.month) for p in months])


EXPIRIES = monthly_expiries()
assert all(d.weekday() == 4 and 15 <= d.day <= 21 for d in EXPIRIES)
EXPIRIES[:6]

DatetimeIndex(['2026-01-16', '2026-02-20', '2026-03-20', '2026-04-17',
               '2026-05-15', '2026-06-19'],
              dtype='datetime64[ns]', freq=None)

In [3]:
def simulate_spot_paths():
    rows = []
    dt = 1 / 252
    for symbol, p in UNDERLYINGS.iterrows():
        shocks = rng.normal(size=len(trading_days))
        log_returns = (-0.5 * p.annual_vol**2) * dt + p.annual_vol * np.sqrt(dt) * shocks
        spots = p.start_spot * np.exp(np.cumsum(log_returns))
        rows.extend(zip([symbol] * len(trading_days), trading_days, spots))
    return pd.DataFrame(rows, columns=['symbol', 'trade_date', 'spot'])


spot_panel = simulate_spot_paths().set_index(['symbol', 'trade_date'])['spot']

# Higher activity near the open and close; uniform within each interval.
TIME_SEGMENTS = pd.DataFrame({
    'segment': ['open', 'midday', 'close'],
    'start_minute': [0, 60, 330],
    'end_minute': [60, 330, 390],
    'weight': [0.30, 0.40, 0.30],
}).set_index('segment')

MATURITY_BUCKETS = pd.DataFrame({
    'maturity_bucket': ['7-30d', '31-60d', '61-120d', '121-240d', '241-420d'],
    'min_dte': [7, 31, 61, 121, 241],
    'max_dte': [30, 60, 120, 240, 420],
    'weight': [0.31, 0.27, 0.22, 0.14, 0.06],
}).set_index('maturity_bucket')

MONEYNESS_BUCKETS = pd.DataFrame({
    'moneyness_bucket': ['deep_put', 'put_wing', 'near_atm', 'call_wing', 'deep_call'],
    'low': [0.70, 0.85, 0.97, 1.03, 1.15],
    'high': [0.85, 0.97, 1.03, 1.15, 1.30],
    'weight': [0.07, 0.20, 0.46, 0.20, 0.07],
}).set_index('moneyness_bucket')

# Public-data proxy: many very small trades and a thin institutional-size tail.
SIZE_BUCKETS = pd.DataFrame({
    'size_bucket': ['1', '2-5', '6-20', '21-99', '100-199', '200-500'],
    'low': [1, 2, 6, 21, 100, 200],
    'high': [1, 5, 20, 99, 199, 500],
    'weight': [0.55, 0.34, 0.08, 0.025, 0.004, 0.001],
}).set_index('size_bucket')

pd.DataFrame({
    'maturity_weight': MATURITY_BUCKETS.weight,
}).dropna().T

maturity_bucket,7-30d,31-60d,61-120d,121-240d,241-420d
maturity_weight,0.31,0.27,0.22,0.14,0.06


In [4]:
def choose_expiry(trade_date, bucket):
    spec = MATURITY_BUCKETS.loc[bucket]
    dtes = (EXPIRIES - trade_date).days
    valid = (dtes >= spec.min_dte) & (dtes <= spec.max_dte)
    candidates = EXPIRIES[valid]
    candidate_dtes = dtes[valid]
    if len(candidates) == 0:
        candidates = EXPIRIES[dtes > 0]
        candidate_dtes = dtes[dtes > 0]
    midpoint = (spec.min_dte + spec.max_dte) / 2
    weights = np.asarray(np.exp(-np.abs(candidate_dtes - midpoint) / 35), dtype=float)
    return rng.choice(candidates, p=weights / weights.sum())


def strike_increment(spot):
    if spot < 75:
        return 1.0
    if spot < 200:
        return 2.5
    return 5.0


def draw_contracts(size_bucket):
    low, high = SIZE_BUCKETS.loc[size_bucket, ['low', 'high']]
    if low == high:
        return int(low)
    # Log-uniform within buckets keeps observations near each bucket's lower edge.
    return int(np.clip(np.floor(np.exp(rng.uniform(np.log(low), np.log(high + 1)))), low, high))


def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black_scholes(spot, strike, years, vol, option_type, rate=0.04):
    years = max(years, 1 / 365)
    vol = max(vol, 0.01)
    d1 = (math.log(spot / strike) + (rate + 0.5 * vol**2) * years) / (vol * math.sqrt(years))
    d2 = d1 - vol * math.sqrt(years)
    if option_type == 'C':
        price = spot * norm_cdf(d1) - strike * math.exp(-rate * years) * norm_cdf(d2)
        delta = norm_cdf(d1)
    else:
        price = strike * math.exp(-rate * years) * norm_cdf(-d2) - spot * norm_cdf(-d1)
        delta = norm_cdf(d1) - 1
    return max(price, 0.01), delta


def generate_trades(n_trades=N_TRADES):
    symbols = rng.choice(UNDERLYINGS.index, n_trades, p=UNDERLYINGS.volume_weight)
    dates = rng.choice(trading_days, n_trades)
    maturity = rng.choice(MATURITY_BUCKETS.index, n_trades, p=MATURITY_BUCKETS.weight)
    money = rng.choice(MONEYNESS_BUCKETS.index, n_trades, p=MONEYNESS_BUCKETS.weight)
    size_buckets = rng.choice(SIZE_BUCKETS.index, n_trades, p=SIZE_BUCKETS.weight)
    segments = rng.choice(TIME_SEGMENTS.index, n_trades, p=TIME_SEGMENTS.weight)
    option_types = rng.choice(['C', 'P'], n_trades)
    sides = rng.choice(['Buy', 'Sell'], n_trades)

    records = []
    for i in range(n_trades):
        symbol = symbols[i]
        trade_date = pd.Timestamp(dates[i])
        spot = float(spot_panel.loc[(symbol, trade_date)])
        expiry = pd.Timestamp(choose_expiry(trade_date, maturity[i]))
        dte = (expiry - trade_date).days

        m_spec = MONEYNESS_BUCKETS.loc[money[i]]
        strike_ratio = rng.uniform(m_spec.low, m_spec.high)
        increment = strike_increment(spot)
        strike = max(increment, round((spot * strike_ratio) / increment) * increment)

        segment = TIME_SEGMENTS.loc[segments[i]]
        minute = int(rng.integers(segment.start_minute, segment.end_minute))
        second = int(rng.integers(0, 60))
        timestamp = trade_date + pd.Timedelta(hours=9, minutes=30 + minute, seconds=second)

        vol = float(UNDERLYINGS.loc[symbol, 'annual_vol'])
        smile_vol = vol * (1 + 0.8 * abs(math.log(strike / spot)))
        option_price, delta = black_scholes(spot, strike, dte / 365, smile_vol, option_types[i])
        contracts = draw_contracts(size_buckets[i])

        component_weight = (
            UNDERLYINGS.loc[symbol, 'volume_weight']
            * MATURITY_BUCKETS.loc[maturity[i], 'weight']
            * MONEYNESS_BUCKETS.loc[money[i], 'weight']
            * 0.5 * 0.5
        )
        side_sign = 1 if sides[i] == 'Buy' else -1
        records.append({
            'trade_id': f'T{i + 1:07d}',
            'timestamp_et': timestamp,
            'trade_date': trade_date,
            'symbol': symbol,
            'expiry': expiry,
            'dte': dte,
            'maturity_bucket': maturity[i],
            'option_type': option_types[i],
            'strike': strike,
            'spot': round(spot, 4),
            'strike_over_spot': round(strike / spot, 5),
            'moneyness_bucket': money[i],
            'side': sides[i],
            'contracts': contracts,
            'size_bucket': size_buckets[i],
            'time_segment': segments[i],
            'implied_vol_proxy': round(smile_vol, 6),
            'option_price': round(option_price, 4),
            'delta': round(delta, 6),
            'signed_delta_contracts': round(side_sign * delta * contracts * 100, 4),
            'sampling_weight': component_weight,
        })

    trades = pd.DataFrame(records).sort_values('timestamp_et').reset_index(drop=True)
    trades['option_line'] = (
        trades['symbol'] + '_' + trades['expiry'].dt.strftime('%Y%m%d') + '_'
        + trades['option_type'] + '_' + trades['strike'].map(lambda x: f'{x:g}')
    )
    line_volume = trades.groupby('option_line')['contracts'].transform('sum')
    trades['line_contract_volume'] = line_volume
    trades['line_volume_share'] = line_volume / trades['contracts'].sum()
    return trades


trades = generate_trades()
trades.head()

,trade_id,timestamp_et,trade_date,symbol,expiry,dte,maturity_bucket,option_type,strike,spot,...,size_bucket,time_segment,implied_vol_proxy,option_price,delta,signed_delta_contracts,sampling_weight,option_line,line_contract_volume,line_volume_share
0,T0020566,2026-01-02 09:30:03,2026-01-02,NVDA,2026-01-16,14,7-30d,P,147.5,148.3554,...,2-5,open,0.482221,5.0430,-0.450382,135.1145,0.006061,NVDA_20260116_P_147.5,6,0.000026
1,T0032244,2026-01-02 09:30:19,2026-01-02,QQQ,2026-03-20,77,61-120d,C,405.0,519.7125,...,2-5,open,0.263892,118.4474,0.985657,492.8285,0.000770,QQQ_20260320_C_405,8,0.000034
2,T0009970,2026-01-02 09:30:23,2026-01-02,QQQ,2026-06-19,168,121-240d,C,510.0,519.7125,...,2-5,open,0.223320,41.2428,0.626193,-125.2386,0.003220,QQQ_20260619_C_510,12,0.000052
3,T0038355,2026-01-02 09:30:47,2026-01-02,SPY,2026-03-20,77,61-120d,P,585.0,594.4969,...,2-5,open,0.182319,13.2436,-0.368834,-73.7667,0.007590,SPY_20260320_P_585,130,0.000559
4,T0022784,2026-01-02 09:31:07,2026-01-02,SPY,2026-01-16,14,7-30d,P,610.0,594.4969,...,2-5,open,0.183707,17.8055,-0.743731,-297.4926,0.010695,SPY_20260116_P_610,18,0.000077


In [5]:
# Validation checks for the requested scope and distributions.
assert trades.trade_date.min() >= START_DATE
assert trades.trade_date.max() <= END_DATE
assert trades.expiry.map(lambda x: x.weekday() == 4 and 15 <= x.day <= 21).all()
assert (trades.expiry > trades.trade_date).all()
assert trades.timestamp_et.dt.time.min() >= pd.Timestamp('09:30').time()
assert trades.timestamp_et.dt.time.max() < pd.Timestamp('16:00').time()

summary = {
    'rows': len(trades),
    'date_range': (trades.trade_date.min().date(), trades.trade_date.max().date()),
    'unique_option_lines': trades.option_line.nunique(),
    'buy_trade_share': round((trades.side == 'Buy').mean(), 4),
    'one_contract_trade_share': round((trades.contracts == 1).mean(), 4),
    'one_contract_volume_share': round(
        trades.loc[trades.contracts == 1, 'contracts'].sum() / trades.contracts.sum(), 4
    ),
    'median_contracts': float(trades.contracts.median()),
    'mean_contracts': round(trades.contracts.mean(), 2),
    'max_contracts': int(trades.contracts.max()),
}
summary

{'rows': 50000,
 'date_range': (datetime.date(2026, 1, 2), datetime.date(2026, 5, 29)),
 'unique_option_lines': 7502,
 'buy_trade_share': np.float64(0.4976),
 'one_contract_trade_share': np.float64(0.5536),
 'one_contract_volume_share': np.float64(0.1191),
 'median_contracts': 1.0,
 'mean_contracts': np.float64(4.65),
 'max_contracts': 487}

In [6]:
distribution_checks = {
    'symbol_trade_share': trades.symbol.value_counts(normalize=True).sort_index(),
    'maturity_trade_share': trades.maturity_bucket.value_counts(normalize=True),
    'moneyness_trade_share': trades.moneyness_bucket.value_counts(normalize=True),
    'side_trade_share': trades.side.value_counts(normalize=True),
    'time_segment_trade_share': trades.time_segment.value_counts(normalize=True),
    'size_bucket_trade_share': trades.size_bucket.value_counts(normalize=True),
}
for name, values in distribution_checks.items():
    print(f'\n{name}')
    print(values.round(4).to_string())


symbol_trade_share
symbol
AAPL    0.1303
JPM     0.0996
MSFT    0.1017
NVDA    0.1694
QQQ     0.1980
SPY     0.3010

maturity_trade_share
maturity_bucket
7-30d       0.3087
31-60d      0.2743
61-120d     0.2190
121-240d    0.1365
241-420d    0.0615

moneyness_trade_share
moneyness_bucket
near_atm     0.4611
call_wing    0.2010
put_wing     0.1994
deep_call    0.0698
deep_put     0.0686

side_trade_share
side
Sell    0.5024
Buy     0.4976

time_segment_trade_share
time_segment
midday    0.3988
open      0.3011
close     0.3001

size_bucket_trade_share
size_bucket
1          0.5536
2-5        0.3377
6-20       0.0791
21-99      0.0247
100-199    0.0039
200-500    0.0010


In [7]:
trades.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(trades):,} rows to {OUTPUT_CSV}')
trades.sample(10, random_state=SEED).sort_values('timestamp_et')

Wrote 50,000 rows to D:\Desktop\Berkeley\Industry Project\JP Morgan\data\simulated\synthetic_option_trades_2026.csv


,trade_id,timestamp_et,trade_date,symbol,expiry,dte,maturity_bucket,option_type,strike,spot,...,size_bucket,time_segment,implied_vol_proxy,option_price,delta,signed_delta_contracts,sampling_weight,option_line,line_contract_volume,line_volume_share
3324,T0032387,2026-01-12 14:31:49,2026-01-12,QQQ,2026-03-20,67,61-120d,C,515.0,519.6842,...,1,midday,0.221594,24.0319,0.587134,-58.7134,0.005060,QQQ_20260320_C_515,38,0.000163
4981,T0010820,2026-01-15 15:53:07,2026-01-15,QQQ,2026-11-20,309,241-420d,P,505.0,495.8032,...,2-5,close,0.223235,36.5491,-0.429329,171.7316,0.001380,QQQ_20261120_P_505,5,0.000022
10296,T0015162,2026-02-02 15:07:03,2026-02-02,SPY,2026-08-21,200,121-240d,P,580.0,579.2697,...,1,close,0.180181,24.9274,-0.412323,-41.2323,0.004830,SPY_20260821_P_580,155,0.000667
11891,T0046551,2026-02-05 15:23:06,2026-02-05,NVDA,2026-02-20,15,7-30d,C,112.5,120.0483,...,2-5,close,0.504937,9.5770,0.758557,227.5670,0.002635,NVDA_20260220_C_112.5,30,0.000129
14859,T0018401,2026-02-17 09:51:11,2026-02-17,QQQ,2026-12-18,304,241-420d,P,485.0,489.2040,...,1,open,0.221519,29.3555,-0.378826,37.8826,0.001380,QQQ_20261218_P_485,4,0.000017
17408,T0022979,2026-02-24 09:43:33,2026-02-24,MSFT,2026-05-15,80,61-120d,P,400.0,407.7331,...,1,open,0.253830,13.9628,-0.384254,-38.4254,0.002530,MSFT_20260515_P_400,199,0.000856
27309,T0011560,2026-03-24 09:52:44,2026-03-24,QQQ,2026-06-19,87,61-120d,P,430.0,440.2234,...,1,open,0.224136,12.5979,-0.360706,36.0706,0.005060,QQQ_20260619_P_430,55,0.000237
33468,T0024386,2026-04-10 15:17:51,2026-04-10,SPY,2026-04-17,7,7-30d,C,675.0,570.6277,...,1,close,0.204188,0.0100,0.000000,-0.0000,0.001628,SPY_20260417_C_675,63,0.000271
38235,T0011512,2026-04-24 15:10:10,2026-04-24,NVDA,2026-08-21,119,61-120d,C,65.0,90.5657,...,6-20,close,0.607368,28.6144,0.878481,966.3294,0.000655,NVDA_20260821_C_65,13,0.000056
40221,T0009073,2026-04-30 15:52:23,2026-04-30,NVDA,2026-05-15,15,7-30d,P,100.0,99.7506,...,2-5,close,0.480959,3.9229,-0.484048,193.6192,0.006061,NVDA_20260515_P_100,69,0.000297


## Interpretation and limitations

- `sampling_weight` records the ex-ante product of the symbol, maturity, moneyness, option-type, and side probabilities used for each trade.
- `line_contract_volume` and `line_volume_share` are realized contract-volume weights for each specific option line.
- Buy and Sell are sampled independently at 50/50; small deviations in the realized dataset are expected.
- The intraday distribution assigns 30% of trades to the first hour, 40% to midday, and 30% to the final hour.
- Prices, implied volatilities, and deltas are synthetic Black-Scholes proxies. They are included for delta-hedging experiments, not as reconstructed 2026 market observations.
- The size distribution is a reproducible stylization based on public evidence and published size categories. It should be sensitivity-tested because free public sources do not provide a complete historical trade-by-trade options tape.